# Módulo 1 · Clase 9 — Excepciones

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Bloque | Qué aprenderás |
|---|--------|----------------|
| 1 | ¿Qué es una excepción? | Errores en tiempo de ejecución y por qué no hay que temerlos |
| 2 | `try` / `except` | Capturar y manejar errores |
| 3 | Tipos de excepciones | Las más comunes y cuándo aparecen |
| 4 | `else` y `finally` | Código que se ejecuta con o sin error |
| 5 | `raise` | Lanzar excepciones propias |
| 6 | Excepciones personalizadas | Crear tus propias clases de error |
| 7 | Buenas prácticas | Qué capturar y qué no |
| 8 | Práctica | Validador de datos |

> 💡 **Idea central:** en el mundo real, los programas fallan — el usuario introduce datos incorrectos, el archivo no existe, la red se cae. Una excepción no es un fracaso del programador: es la forma que tiene Python de decir «algo inesperado ocurrió». Nuestro trabajo es decidir qué hacer cuando eso pasa.

---
## 1. ¿Qué es una excepción?

Cuando Python encuentra un error en tiempo de ejecución, **lanza una excepción**: detiene el programa y muestra un mensaje de error llamado **traceback**.

```
Traceback (most recent call last):
  File "programa.py", line 3, in <module>
    resultado = 10 / 0
ZeroDivisionError: division by zero
```

El traceback tiene tres partes:
1. **Dónde ocurrió** — archivo y número de línea
2. **Qué línea causó el error** — el código
3. **Qué tipo de error es** — el nombre de la excepción y un mensaje

### Diferencia entre errores de sintaxis y excepciones

| Tipo | Cuándo aparece | Ejemplo |
|------|---------------|--------|
| `SyntaxError` | Antes de ejecutar (Python no puede leer el código) | `if True` sin `:` |
| Excepción | Durante la ejecución (el código es válido pero algo falla) | `10 / 0` |

Las excepciones **sí se pueden capturar y manejar**. Los errores de sintaxis no.

In [ ]:
# Descomenta cada línea para ver qué excepción lanza

# 10 / 0                     # ZeroDivisionError
# int("hola")                # ValueError
# lista = [1,2,3]; lista[9]  # IndexError
# d = {}; d["clave"]         # KeyError
# None + 1                   # TypeError
# open("no_existe.txt")      # FileNotFoundError
# variable_que_no_existe     # NameError

print("Ejecuta una línea a la vez para ver el traceback")

---
## 2. `try` / `except` — Capturar excepciones

La estructura `try/except` permite «intentar» ejecutar código y decidir qué hacer si falla, en lugar de que el programa se rompa.

```python
try:
    # código que podría fallar
except TipoDeError:
    # qué hacer si ese error ocurre
```

**Flujo de ejecución:**
```
Entra en try
   │
   ├─── Si todo va bien ──► sale del try/except y continúa
   │
   └─── Si ocurre un error ──► salta al except correspondiente
                                     │
                                     └─► continúa después del try/except
```

In [ ]:
# Sin manejo de errores — el programa se rompe
# resultado = int("abc")   # ValueError: detiene el programa

# Con try/except — el programa continúa
try:
    resultado = int("abc")
    print(f"Resultado: {resultado}")   # esta línea NO se ejecuta
except ValueError:
    print("Error: no se puede convertir 'abc' a entero")

print("El programa continúa aquí")

In [ ]:
# Acceder al mensaje de error con 'as'
try:
    resultado = 10 / 0
except ZeroDivisionError as e:
    print(f"Tipo de error:    {type(e).__name__}")
    print(f"Mensaje de error: {e}")
    print(f"Objeto completo:  {repr(e)}")

In [ ]:
# Capturar múltiples tipos de error
def convertir_a_entero(valor):
    """Convierte un valor a entero, manejando los errores posibles."""
    try:
        return int(valor)
    except ValueError:
        print(f"ValueError: '{valor}' no es un número válido")
        return None
    except TypeError:
        print(f"TypeError: no se puede convertir {type(valor).__name__} a entero")
        return None

print(convertir_a_entero("42"))     # funciona
print(convertir_a_entero("hola"))   # ValueError
print(convertir_a_entero(None))     # TypeError
print(convertir_a_entero(3.7))      # funciona (trunca)

In [ ]:
# Capturar varios errores con una sola cláusula (tupla)
def leer_numero(texto):
    try:
        return float(texto)
    except (ValueError, TypeError) as e:
        print(f"No se pudo convertir: {e}")
        return None

print(leer_numero("3.14"))
print(leer_numero("abc"))
print(leer_numero(None))

---
## 3. Jerarquía y tipos de excepciones

Todas las excepciones en Python heredan de `BaseException`. Las que usarás en el día a día heredan de `Exception`.

```
BaseException
└── Exception
    ├── ValueError        → valor incorrecto (int("abc"))
    ├── TypeError         → tipo incorrecto ("texto" + 5)
    ├── KeyError          → clave no existe en dict
    ├── IndexError        → índice fuera de rango
    ├── AttributeError    → atributo no existe (None.upper())
    ├── ZeroDivisionError → división entre cero
    ├── FileNotFoundError → archivo no encontrado
    ├── PermissionError   → sin permisos de acceso
    ├── ImportError       → módulo no encontrado
    ├── NameError         → variable no definida
    ├── OverflowError     → número demasiado grande
    └── RuntimeError      → error genérico de ejecución
```

Capturar `Exception` captura casi cualquier error. Capturar `BaseException` captura también `KeyboardInterrupt` (Ctrl+C) y `SystemExit` — casi nunca queremos eso.

In [ ]:
# Catálogo de excepciones comunes
casos = [
    (lambda: 1 / 0,              "ZeroDivisionError"),
    (lambda: int("hola"),        "ValueError"),
    (lambda: "a" + 1,            "TypeError"),
    (lambda: {}["clave"],        "KeyError"),
    (lambda: [][0],              "IndexError"),
    (lambda: None.upper(),       "AttributeError"),
    (lambda: open("___x___"),    "FileNotFoundError"),
]

for funcion, nombre_esperado in casos:
    try:
        funcion()
    except Exception as e:
        tipo = type(e).__name__
        icono = "✅" if tipo == nombre_esperado else "❌"
        print(f"{icono} {tipo:<22} → {e}")

---
## 4. `else` y `finally` — Control completo del flujo

La estructura completa de manejo de excepciones tiene cuatro partes:

```python
try:
    # código que puede fallar
except TipoError:
    # solo si ocurre ese error
else:
    # solo si NO ocurrió ningún error
finally:
    # SIEMPRE, haya o no error
```

| Bloque | Se ejecuta |
|--------|------------|
| `try` | Siempre (es el intento) |
| `except` | Solo si ocurrió el error indicado |
| `else` | Solo si **no** ocurrió ningún error |
| `finally` | **Siempre**, con o sin error — ideal para liberar recursos |

In [ ]:
def dividir(a, b):
    try:
        resultado = a / b
    except ZeroDivisionError:
        print("  [except]  División por cero")
        resultado = None
    else:
        print(f"  [else]    Todo bien, resultado = {resultado}")
    finally:
        print(f"  [finally] Siempre se ejecuta")
    return resultado

print("--- Caso 1: sin error ---")
dividir(10, 2)

print("\n--- Caso 2: con error ---")
dividir(10, 0)

In [ ]:
# finally es imprescindible para liberar recursos
# (aunque con 'with' esto se hace automático para archivos)

def procesar_archivo(ruta):
    archivo = None
    try:
        archivo = open(ruta, "r", encoding="utf-8")
        contenido = archivo.read()
        lineas = contenido.splitlines()
        print(f"Leídas {len(lineas)} líneas")
        return lineas
    except FileNotFoundError:
        print(f"Archivo no encontrado: {ruta}")
        return []
    finally:
        if archivo:                # si se abrió, hay que cerrarlo
            archivo.close()
            print("Archivo cerrado")

# Aunque con 'with' es más limpio:
def procesar_archivo_v2(ruta):
    try:
        with open(ruta, "r", encoding="utf-8") as f:
            return f.read().splitlines()
    except FileNotFoundError:
        print(f"Archivo no encontrado: {ruta}")
        return []

procesar_archivo("M1_C9_Excepciones.ipynb")   # existe
print()
procesar_archivo("no_existe.txt")              # no existe

---
## 5. `raise` — Lanzar excepciones

Además de capturar excepciones, podemos **lanzarlas nosotros mismos** con `raise`. Esto es útil para:
- Validar datos de entrada al principio de una función.
- Propagar un error añadiendo más contexto.
- Señalar condiciones de negocio que no son errores de Python pero sí de nuestra lógica.

```python
raise TipoDeError("mensaje descriptivo")
```

In [ ]:
def calcular_raiz(numero):
    """Calcula la raíz cuadrada. Lanza ValueError si el número es negativo."""
    if numero < 0:
        raise ValueError(f"No se puede calcular la raíz de un número negativo: {numero}")
    return numero ** 0.5

# Caso válido
print(calcular_raiz(16))

# Caso inválido — capturamos la excepción que lanzamos
try:
    print(calcular_raiz(-4))
except ValueError as e:
    print(f"Error capturado: {e}")

In [ ]:
# Relanzar una excepción añadiendo contexto
def cargar_configuracion(ruta):
    import json
    try:
        with open(ruta, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"No se encontró el archivo de configuración: '{ruta}'. "
            "Asegúrate de que existe antes de arrancar la aplicación."
        )
    except json.JSONDecodeError as e:
        raise ValueError(
            f"El archivo '{ruta}' no es un JSON válido. Error en línea {e.lineno}: {e.msg}"
        ) from e   # 'from e' encadena la excepción original

try:
    config = cargar_configuracion("no_existe.json")
except FileNotFoundError as e:
    print(f"FileNotFoundError: {e}")

---
## 6. Excepciones personalizadas

Puedes crear tus propias clases de excepción heredando de `Exception`. Esto permite:
- Dar nombres significativos a los errores de tu dominio.
- Añadir atributos extra (campo que falló, valor incorrecto, etc.).
- Capturar solo tus errores sin interceptar los de Python.

```python
class MiError(Exception):
    pass
```

In [ ]:
# Excepción simple — solo el nombre ya aporta contexto
class EdadInvalidaError(Exception):
    pass

class NombreVacioError(Exception):
    pass

def registrar_usuario(nombre, edad):
    if not nombre or not nombre.strip():
        raise NombreVacioError("El nombre no puede estar vacío")
    if not isinstance(edad, int) or edad < 0 or edad > 150:
        raise EdadInvalidaError(f"Edad no válida: {edad}. Debe ser un entero entre 0 y 150")
    return {"nombre": nombre.strip(), "edad": edad}

casos = [
    ("Ana", 28),
    ("",    25),
    ("Luis", -5),
    ("María", 200)
]

for nombre, edad in casos:
    try:
        usuario = registrar_usuario(nombre, edad)
        print(f"✅ Registrado: {usuario}")
    except NombreVacioError as e:
        print(f"❌ NombreVacioError: {e}")
    except EdadInvalidaError as e:
        print(f"❌ EdadInvalidaError: {e}")

In [ ]:
# Excepción enriquecida con atributos extra
class ErrorValidacion(Exception):
    """Excepción para errores de validación de datos."""
    def __init__(self, campo, valor, mensaje):
        self.campo   = campo
        self.valor   = valor
        self.mensaje = mensaje
        super().__init__(f"[{campo}] {mensaje} (valor recibido: {repr(valor)})")

def validar_email(email):
    if not isinstance(email, str):
        raise ErrorValidacion("email", email, "debe ser un string")
    if "@" not in email:
        raise ErrorValidacion("email", email, "debe contener '@'")
    if email.count("@") > 1:
        raise ErrorValidacion("email", email, "solo puede contener un '@'")
    return email.lower().strip()

casos_email = ["ana@mail.com", "sin_arroba", 123, "dos@@mail.com"]

for email in casos_email:
    try:
        resultado = validar_email(email)
        print(f"✅ Email válido: {resultado}")
    except ErrorValidacion as e:
        print(f"❌ {e}")
        print(f"   Campo: {e.campo}, Valor: {e.valor}")

### Jerarquía de excepciones propias

En proyectos grandes es útil crear una jerarquía: una excepción base para el proyecto y subclases para casos específicos.

In [ ]:
# Jerarquía de excepciones para una app
class AppError(Exception):
    """Base para todos los errores de la aplicación."""

class ErrorDatos(AppError):
    """Errores relacionados con los datos."""

class ErrorArchivo(AppError):
    """Errores relacionados con archivos."""

class ErrorFormatoCSV(ErrorDatos):
    """El CSV no tiene el formato esperado."""

class ErrorColumnaFaltante(ErrorDatos):
    """Una columna requerida no está en el CSV."""

# Capturar por nivel de especificidad
def procesar(tipo_error):
    if tipo_error == "columna":
        raise ErrorColumnaFaltante("Falta la columna 'precio'")
    elif tipo_error == "formato":
        raise ErrorFormatoCSV("El separador no es una coma")
    elif tipo_error == "archivo":
        raise ErrorArchivo("El archivo no existe")

for tipo in ["columna", "formato", "archivo"]:
    try:
        procesar(tipo)
    except ErrorColumnaFaltante as e:
        print(f"Columna faltante (específico): {e}")
    except ErrorDatos as e:
        print(f"Error de datos (general): {e}")
    except AppError as e:
        print(f"Error de app (base): {e}")

---
## 7. Buenas prácticas

### ✅ Lo que hay que hacer

```python
# 1. Captura excepciones específicas, no Exception genérico
try:
    valor = int(texto)
except ValueError:        # ← específico
    valor = 0

# 2. Usa 'as e' para acceder al mensaje
except ValueError as e:
    print(f"Error: {e}")

# 3. Captura lo más cerca posible del error
# 4. No dejes except vacíos — siempre haz algo
# 5. Usa finally para liberar recursos
```

### ❌ Lo que hay que evitar

In [ ]:
# ❌ ANTIPATRÓN 1: capturar todo sin hacer nada (silenciar errores)
try:
    resultado = int("abc")
except Exception:
    pass   # ← el error desaparece en silencio, muy peligroso

# ✅ Mínimo: loguear el error
try:
    resultado = int("abc")
except ValueError as e:
    print(f"WARN: conversión fallida — {e}")
    resultado = 0

# ❌ ANTIPATRÓN 2: usar excepciones para control de flujo normal
def es_entero_v1(texto):
    try:
        int(texto)
        return True
    except ValueError:
        return False

# ✅ Mejor: comprobar primero con el método adecuado
def es_entero_v2(texto):
    return texto.strip().lstrip("-").isdigit()

for t in ["42", "abc", "-7", "3.14"]:
    print(f"'{t}': v1={es_entero_v1(t)}  v2={es_entero_v2(t)}")

### El patrón EAFP vs LBYL

Python tiene dos filosofías de manejo de errores:

| Estilo | Nombre | Descripción | Cuándo |
|--------|--------|-------------|--------|
| **EAFP** | Easier to Ask Forgiveness than Permission | Intenta la operación y captura si falla | Cuando el caso normal es el más frecuente |
| **LBYL** | Look Before You Leap | Comprueba antes de hacer | Cuando la comprobación es barata y clara |

In [ ]:
datos = {"nombre": "Ana", "edad": 28}

# LBYL — comprueba antes
if "email" in datos:
    email = datos["email"]
else:
    email = "sin email"

# EAFP — intenta y captura
try:
    email = datos["email"]
except KeyError:
    email = "sin email"

# En Python: .get() es la forma más idiomática para este caso
email = datos.get("email", "sin email")

print(f"Email: {email}")

---
## 8. Práctica — Validador de datos

Construimos un sistema de validación robusto que usa excepciones personalizadas para comunicar exactamente qué falló en los datos de entrada. Es el tipo de código que encontrarás en cualquier aplicación real que reciba datos de usuarios o archivos externos.

---

### Preparación — Excepciones base del validador

Definimos la jerarquía de excepciones que usaremos en todos los ejercicios.

In [ ]:
# Jerarquía de excepciones del validador
class ErrorValidacion(Exception):
    """Base para todos los errores de validación."""
    def __init__(self, campo, valor, mensaje):
        self.campo   = campo
        self.valor   = valor
        self.mensaje = mensaje
        super().__init__(f"[{campo}] {mensaje}")

class ErrorTipo(ErrorValidacion):
    """El valor tiene un tipo incorrecto."""

class ErrorRango(ErrorValidacion):
    """El valor está fuera del rango permitido."""

class ErrorFormato(ErrorValidacion):
    """El valor no cumple el formato esperado."""

class ErrorRequerido(ErrorValidacion):
    """Un campo obligatorio está vacío o es None."""
    def __init__(self, campo):
        super().__init__(campo, None, "campo obligatorio vacío o ausente")

print("✅ Jerarquía de excepciones definida")

---
### Ejercicio 1 — Validadores de campo individuales

**Enunciado:** Escribe funciones que validen un campo y lancen la excepción adecuada si falla:

1. `validar_texto(campo, valor, min_len=1, max_len=None)` — no vacío, longitud entre límites.
2. `validar_entero(campo, valor, minimo=None, maximo=None)` — convertible a `int`, dentro del rango.
3. `validar_decimal(campo, valor, minimo=None, maximo=None)` — convertible a `float`, dentro del rango.
4. `validar_email(campo, valor)` — contiene `@`, tiene dominio con punto.

Cada función debe devolver el valor ya convertido al tipo correcto si la validación pasa.

**Ejemplo:**
```python
validar_entero("edad", "28", minimo=0, maximo=150)  →  28
validar_entero("edad", "abc", minimo=0, maximo=150) →  lanza ErrorTipo
validar_entero("edad", "-5", minimo=0, maximo=150)  →  lanza ErrorRango
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
def validar_texto(campo, valor, min_len=1, max_len=None):
    """Valida que el valor sea un string no vacío dentro de la longitud indicada."""
    if not isinstance(valor, str):
        raise ErrorTipo(campo, valor, f"se esperaba str, llegó {type(valor).__name__}")
    valor = valor.strip()
    if len(valor) < min_len:
        raise ErrorRequerido(campo) if min_len == 1 else \
              ErrorRango(campo, valor, f"mínimo {min_len} caracteres, tiene {len(valor)}")
    if max_len is not None and len(valor) > max_len:
        raise ErrorRango(campo, valor, f"máximo {max_len} caracteres, tiene {len(valor)}")
    return valor

def validar_entero(campo, valor, minimo=None, maximo=None):
    """Convierte a int y valida el rango."""
    try:
        valor_int = int(valor)
    except (ValueError, TypeError):
        raise ErrorTipo(campo, valor, f"no se puede convertir '{valor}' a entero")
    if minimo is not None and valor_int < minimo:
        raise ErrorRango(campo, valor_int, f"mínimo permitido: {minimo}")
    if maximo is not None and valor_int > maximo:
        raise ErrorRango(campo, valor_int, f"máximo permitido: {maximo}")
    return valor_int

def validar_decimal(campo, valor, minimo=None, maximo=None):
    """Convierte a float y valida el rango."""
    try:
        valor_float = float(valor)
    except (ValueError, TypeError):
        raise ErrorTipo(campo, valor, f"no se puede convertir '{valor}' a decimal")
    if minimo is not None and valor_float < minimo:
        raise ErrorRango(campo, valor_float, f"mínimo permitido: {minimo}")
    if maximo is not None and valor_float > maximo:
        raise ErrorRango(campo, valor_float, f"máximo permitido: {maximo}")
    return valor_float

def validar_email(campo, valor):
    """Valida que el valor tenga formato de email básico."""
    if not isinstance(valor, str):
        raise ErrorTipo(campo, valor, "se esperaba str")
    valor = valor.strip().lower()
    if not valor:
        raise ErrorRequerido(campo)
    if valor.count("@") != 1:
        raise ErrorFormato(campo, valor, "debe contener exactamente un '@'")
    local, dominio = valor.split("@")
    if not local:
        raise ErrorFormato(campo, valor, "falta la parte antes del '@'")
    if "." not in dominio or len(dominio) < 3:
        raise ErrorFormato(campo, valor, "dominio inválido (falta punto o demasiado corto)")
    return valor

# Pruebas
pruebas = [
    (validar_texto,   ("nombre", "Ana"),              {}),
    (validar_texto,   ("nombre", ""),                 {}),
    (validar_entero,  ("edad", "28"),      {"minimo": 0, "maximo": 150}),
    (validar_entero,  ("edad", "abc"),     {"minimo": 0, "maximo": 150}),
    (validar_entero,  ("edad", "-5"),      {"minimo": 0, "maximo": 150}),
    (validar_decimal, ("precio", "19.99"), {"minimo": 0}),
    (validar_email,   ("email", "ana@mail.com"), {}),
    (validar_email,   ("email", "sin_arroba"),   {}),
]

for func, args, kwargs in pruebas:
    try:
        resultado = func(*args, **kwargs)
        print(f"✅ {func.__name__}({args[1]!r}) → {resultado!r}")
    except ErrorValidacion as e:
        print(f"❌ {type(e).__name__}: {e}")
```

**Puntos clave:**
- Cada función hace una sola cosa y lanza la excepción más específica posible.
- El `try/except (ValueError, TypeError)` en `validar_entero` captura tanto strings no numéricos como `None`.
- Las funciones devuelven el valor **ya convertido** — el código que las llama no necesita volver a convertir.
- `raise ErrorTipo(...) from None` suprimir el contexto original si no queremos encadenar excepciones.
</details>

---
### Ejercicio 2 — Validador de formulario completo

**Enunciado:** Escribe una función `validar_formulario(datos)` que recibe un diccionario con los datos de un usuario y valida todos los campos usando las funciones del ejercicio anterior.

El esquema esperado:
```python
{
    "nombre":   str,   # 2-50 caracteres
    "apellido": str,   # 2-50 caracteres
    "edad":     int,   # 0-120
    "email":    str,   # formato email
    "precio":   float  # > 0
}
```

La función debe:
- Validar **todos** los campos, aunque alguno falle — recoge todos los errores.
- Devolver `{"ok": True, "datos": {...}}` si todo es válido.
- Devolver `{"ok": False, "errores": [{"campo": ..., "mensaje": ...}]}` si hay errores.

**Ejemplo:**
```
{'ok': False, 'errores': [
    {'campo': 'nombre', 'mensaje': 'campo obligatorio vacío o ausente'},
    {'campo': 'edad',   'mensaje': 'máximo permitido: 120'}
]}
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
def validar_formulario(datos):
    """Valida todos los campos del formulario. Recoge todos los errores."""
    errores      = []
    datos_limpios = {}

    # Mapa de validaciones: campo → función con sus argumentos
    validaciones = [
        ("nombre",   validar_texto,   {"min_len": 2, "max_len": 50}),
        ("apellido", validar_texto,   {"min_len": 2, "max_len": 50}),
        ("edad",     validar_entero,  {"minimo": 0,  "maximo": 120}),
        ("email",    validar_email,   {}),
        ("precio",   validar_decimal, {"minimo": 0.01}),
    ]

    for campo, funcion, kwargs in validaciones:
        valor = datos.get(campo)   # None si el campo no existe
        try:
            datos_limpios[campo] = funcion(campo, valor, **kwargs)
        except ErrorValidacion as e:
            errores.append({"campo": e.campo, "tipo": type(e).__name__, "mensaje": e.mensaje})

    if errores:
        return {"ok": False, "errores": errores}
    return {"ok": True, "datos": datos_limpios}

# Prueba 1: datos correctos
datos_ok = {
    "nombre":   "Ana",
    "apellido": "García",
    "edad":     "28",
    "email":    "ana@mail.com",
    "precio":   "99.99"
}
resultado = validar_formulario(datos_ok)
print("Caso 1 (válido):")
print(f"  ok: {resultado['ok']}")
if resultado['ok']:
    for k, v in resultado['datos'].items():
        print(f"  {k}: {v!r} ({type(v).__name__})")

# Prueba 2: varios errores
datos_mal = {
    "nombre":   "",
    "apellido": "García",
    "edad":     "200",
    "email":    "sin_arroba",
    "precio":   "-10"
}
resultado = validar_formulario(datos_mal)
print("\nCaso 2 (con errores):")
print(f"  ok: {resultado['ok']}")
for err in resultado['errores']:
    print(f"  [{err['tipo']}] {err['campo']}: {err['mensaje']}")
```

**Puntos clave:**
- Validamos **todos** los campos aunque alguno falle — acumulamos en `errores` en lugar de parar en el primero. Así el usuario recibe todos los problemas a la vez.
- El mapa `validaciones` separa la configuración de la lógica — añadir un campo nuevo es añadir una línea a la lista.
- `datos.get(campo)` devuelve `None` si el campo no existe — las funciones de validación lo tratarán como `ErrorRequerido`.
- La función devuelve siempre la misma estructura: `{"ok": bool, ...}` — fácil de consumir por quien la llame.
</details>

---
### Ejercicio 3 — Validador de CSV fila a fila

**Enunciado:** Escribe una función `validar_csv(ruta)` que lea un CSV y valide cada fila. Usa el CSV `ventas.csv` de la clase anterior.

Debe:
1. Comprobar que el archivo existe (`FileNotFoundError`).
2. Comprobar que las columnas requeridas están presentes.
3. Validar cada fila: `cantidad` entero ≥ 1, `precio_unitario` decimal ≥ 0.
4. Devolver `{"validas": [...], "invalidas": [{"fila": N, "errores": [...]}]}`.

**Salida esperada:**
```
Filas válidas:   15
Filas inválidas: 0
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import csv
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import csv
from pathlib import Path

COLUMNAS_REQUERIDAS = {"fecha", "producto", "categoria", "cantidad", "precio_unitario", "vendedor"}

def validar_csv(ruta):
    """Lee y valida un CSV de ventas fila a fila."""
    ruta = Path(ruta)

    # Nivel 1: el archivo existe
    if not ruta.exists():
        raise FileNotFoundError(f"El archivo no existe: {ruta}")

    validas   = []
    invalidas = []

    try:
        with open(ruta, "r", encoding="utf-8") as f:
            lector = csv.DictReader(f)

            # Nivel 2: columnas requeridas
            columnas_presentes = set(lector.fieldnames or [])
            faltantes = COLUMNAS_REQUERIDAS - columnas_presentes
            if faltantes:
                raise ErrorValidacion(
                    "cabecera", list(faltantes),
                    f"columnas requeridas ausentes: {sorted(faltantes)}"
                )

            # Nivel 3: validar cada fila
            for num_fila, fila in enumerate(lector, start=2):  # start=2: la 1 es cabecera
                errores_fila = []

                for campo, funcion, kwargs in [
                    ("cantidad",        validar_entero,  {"minimo": 1}),
                    ("precio_unitario", validar_decimal, {"minimo": 0}),
                    ("producto",        validar_texto,   {"min_len": 1, "max_len": 100}),
                    ("vendedor",        validar_texto,   {"min_len": 1}),
                ]:
                    try:
                        fila[campo] = funcion(campo, fila.get(campo), **kwargs)
                    except ErrorValidacion as e:
                        errores_fila.append({"campo": e.campo, "mensaje": e.mensaje})

                if errores_fila:
                    invalidas.append({"fila": num_fila, "datos": dict(fila), "errores": errores_fila})
                else:
                    fila["total"] = round(fila["cantidad"] * fila["precio_unitario"], 2)
                    validas.append(dict(fila))

    except UnicodeDecodeError:
        raise ValueError(f"El archivo '{ruta}' no está codificado en UTF-8")

    return {"validas": validas, "invalidas": invalidas}


# Prueba con el CSV de la clase anterior
ruta_csv = Path("archivos_clase8") / "ventas.csv"

try:
    resultado = validar_csv(ruta_csv)
    print(f"Filas válidas:   {len(resultado['validas'])}")
    print(f"Filas inválidas: {len(resultado['invalidas'])}")
    if resultado['invalidas']:
        for inv in resultado['invalidas']:
            print(f"  Fila {inv['fila']}: {inv['errores']}")
except FileNotFoundError as e:
    print(f"Archivo no encontrado: {e}")
except ErrorValidacion as e:
    print(f"Error de estructura: {e}")
```

**Puntos clave:**
- Tres niveles de validación anidados: archivo → estructura → filas. Cada nivel lanza su propia excepción.
- Acumulamos errores por fila igual que en el ejercicio anterior — el usuario ve todos los problemas.
- `start=2` en `enumerate` porque la fila 1 es la cabecera — los números de fila coinciden con el editor.
- `set(lector.fieldnames or [])` maneja el caso de archivo vacío sin columnas.
</details>

---
### Ejercicio 4 — Sistema de log de errores

**Enunciado:** Escribe un decorador `con_log_errores(funcion)` que envuelva cualquier función y, si lanza una excepción:
1. La registre en `errores.log` con timestamp, tipo de error y mensaje.
2. Devuelva `None` en lugar de propagar el error.
3. Si la función funciona bien, devuelva su resultado normalmente.

**Ejemplo de uso:**
```python
@con_log_errores
def dividir(a, b):
    return a / b

print(dividir(10, 2))   # 5.0
print(dividir(10, 0))   # None (error registrado en log)
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from datetime import datetime


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
from datetime import datetime
from pathlib import Path

LOG_PATH = Path("errores.log")

def con_log_errores(funcion):
    """Decorador que captura excepciones, las registra en log y devuelve None."""
    def envoltorio(*args, **kwargs):
        try:
            return funcion(*args, **kwargs)
        except Exception as e:
            timestamp  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            tipo_error = type(e).__name__
            entrada    = (
                f"[{timestamp}] [{tipo_error}] "
                f"en {funcion.__name__}({args}, {kwargs}): {e}\n"
            )
            with open(LOG_PATH, "a", encoding="utf-8") as f:
                f.write(entrada)
            print(f"⚠️  Error registrado en log: {tipo_error}: {e}")
            return None
    return envoltorio

# Aplicar el decorador
@con_log_errores
def dividir(a, b):
    return a / b

@con_log_errores
def buscar_en_dict(d, clave):
    return d[clave]

@con_log_errores
def convertir(valor):
    return int(valor)

# Pruebas
print(dividir(10, 2))             # 5.0
print(dividir(10, 0))             # None + log
print(buscar_en_dict({"a": 1}, "a"))  # 1
print(buscar_en_dict({"a": 1}, "z"))  # None + log
print(convertir("42"))            # 42
print(convertir("abc"))           # None + log

# Ver el log
print("\n--- Contenido de errores.log ---")
if LOG_PATH.exists():
    print(LOG_PATH.read_text(encoding="utf-8"))
```

**Puntos clave:**
- Un **decorador** es una función que recibe una función y devuelve otra función mejorada — `@con_log_errores` es azúcar sintáctica para `dividir = con_log_errores(dividir)`.
- `*args, **kwargs` en `envoltorio` hace que funcione con cualquier función, sin importar sus parámetros.
- Capturamos `Exception` (no `BaseException`) — dejamos pasar `KeyboardInterrupt` y `SystemExit`.
- El modo `"a"` en el log garantiza que cada error se añade sin borrar los anteriores.
</details>

---
### Ejercicio 5 (reto) — Pipeline de validación con informe

**Enunciado:** Integra todo en una función `pipeline_validacion(ruta_csv, ruta_informe)` que:

1. Intente cargar el CSV — si falla, registra el error y termina con un informe de fallo.
2. Valide todas las filas con `validar_csv`.
3. Procese solo las filas válidas (calcule el total de cada una).
4. Guarde las filas válidas en un nuevo CSV `ventas_limpias.csv`.
5. Genere un informe `.txt` con:
   - Total de filas procesadas, válidas e inválidas.
   - Detalle de cada fila inválida y sus errores.
   - Suma total de las filas válidas.
6. Use `finally` para garantizar que el informe **siempre** se escribe aunque haya un error.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import csv
import json
from pathlib import Path
from datetime import datetime


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import csv
from pathlib import Path
from datetime import datetime

def pipeline_validacion(ruta_csv, ruta_informe):
    """Pipeline completo: carga → valida → guarda → informa."""
    ruta_csv     = Path(ruta_csv)
    ruta_informe = Path(ruta_informe)
    ahora        = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    lineas_informe = [
        "=" * 55,
        f"  INFORME DE VALIDACIÓN — {ahora}",
        "=" * 55,
        f"Archivo: {ruta_csv.name}",
        ""
    ]
    resultado = None

    try:
        # Paso 1: validar el CSV
        resultado = validar_csv(ruta_csv)
        validas   = resultado["validas"]
        invalidas = resultado["invalidas"]
        total     = len(validas) + len(invalidas)

        # Paso 2: calcular el total de las filas válidas
        suma_total = sum(v["total"] for v in validas)

        # Paso 3: guardar filas válidas
        ruta_limpia = ruta_csv.parent / "ventas_limpias.csv"
        if validas:
            with open(ruta_limpia, "w", encoding="utf-8", newline="") as f:
                escritor = csv.DictWriter(f, fieldnames=validas[0].keys())
                escritor.writeheader()
                escritor.writerows(validas)

        # Paso 4: construir informe
        lineas_informe += [
            f"Total filas procesadas: {total}",
            f"Filas válidas:          {len(validas)}",
            f"Filas inválidas:        {len(invalidas)}",
            f"Total facturado:        {suma_total:,.2f} €",
            ""
        ]

        if invalidas:
            lineas_informe.append("--- FILAS CON ERRORES ---")
            for inv in invalidas:
                lineas_informe.append(f"  Fila {inv['fila']}:")
                for err in inv["errores"]:
                    lineas_informe.append(f"    [{err['campo']}] {err['mensaje']}")
        else:
            lineas_informe.append("✅ Todas las filas son válidas.")

        if validas:
            lineas_informe.append(f"\nCSV limpio guardado en: {ruta_limpia.name}")

    except FileNotFoundError as e:
        lineas_informe += [
            "❌ ERROR CRÍTICO: archivo no encontrado",
            str(e)
        ]
    except ErrorValidacion as e:
        lineas_informe += [
            "❌ ERROR CRÍTICO: estructura del CSV no válida",
            str(e)
        ]
    except Exception as e:
        lineas_informe += [
            f"❌ ERROR INESPERADO: {type(e).__name__}",
            str(e)
        ]
    finally:
        # El informe SIEMPRE se escribe, haya error o no
        lineas_informe += ["", "=" * 55]
        contenido = "\n".join(lineas_informe)
        ruta_informe.write_text(contenido, encoding="utf-8")
        print(f"✅ Informe guardado en {ruta_informe}")
        print()
        print(contenido)

# Ejecutar el pipeline
pipeline_validacion(
    Path("archivos_clase8") / "ventas.csv",
    Path("archivos_clase8") / "informe_validacion.txt"
)
```

**Puntos clave:**
- `finally` garantiza que el informe se escriba incluso si hay un error en cualquier paso — fundamental para pipelines de datos donde necesitamos trazabilidad.
- Capturamos excepciones de lo más específico a lo más general: `FileNotFoundError` → `ErrorValidacion` → `Exception`. El orden importa.
- `resultado = None` antes del `try` permite que `finally` sepa si el proceso llegó a completarse.
- Acumular las líneas del informe en una lista y escribir al final (en `finally`) es más robusto que escribir línea a línea.
</details>

---
## Resumen de la clase

### Estructura completa

```python
try:
    resultado = operacion_riesgosa()
except TipoEspecifico as e:    # captura ese tipo concreto
    manejar(e)
except (TypeError, ValueError): # captura varios tipos
    manejar_juntos()
except Exception as e:          # captura cualquier error
    log(e)
else:
    usar(resultado)             # solo si no hubo error
finally:
    liberar_recursos()          # siempre
```

### Lanzar excepciones

```python
raise ValueError("mensaje")          # lanzar
raise MiError(campo, valor, msg)     # excepción propia
raise NuevoError("contexto") from e  # encadenar
```

### Excepciones personalizadas

```python
class MiError(Exception):
    def __init__(self, campo, valor, mensaje):
        self.campo   = campo
        self.valor   = valor
        super().__init__(f"[{campo}] {mensaje}")
```

### Reglas de oro

| Regla | Descripción |
|-------|-------------|
| Específico > general | Captura `ValueError` antes que `Exception` |
| Nunca silencies | Un `except: pass` oculta bugs |
| `finally` para recursos | Garantiza cierre de archivos, conexiones… |
| `raise` con contexto | Añade información al relanzar |
| Valida en las fronteras | Entrada de usuario, archivos, APIs |

---

## ¿Qué viene en la próxima clase?

En la **Clase 10** aprenderemos **Programación Orientada a Objetos (POO)**:
- Clases y objetos
- Atributos y métodos
- Herencia

Las excepciones personalizadas que hemos creado hoy ya usan herencia — en la próxima clase entenderemos exactamente cómo funciona ese mecanismo.